## Import Required Libraries

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate

from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import CTransformers

from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

## Load Environment Variables

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

pinecone_key = os.getenv("PINECONE_API_KEY")
index_name = os.getenv("PINECONE_INDEX_NAME")

from pinecone import Pinecone

pc = Pinecone(api_key=pinecone_key)

index = pc.Index(index_name)

print("Pinecone Connected Successfully")

Pinecone Connected Successfully


In [4]:
print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 4418}},
 'total_vector_count': 4418,
 'vector_type': 'dense'}


## Load PDF Documents

In [5]:
# Extract data from PDF files
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
def load_pdf(data):
    
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyMuPDFLoader
    )

    documents = loader.load()

    return documents

### Load Extracted Data

In [6]:
extracted_data = load_pdf("pdfs/")

FileNotFoundError: Directory not found: 'pdfs/'

In [ ]:
len(extracted_data)

637

## Split Text into Chunks

In [ ]:
def text_split(extracted_data):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )

    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks


text_chunks = text_split(extracted_data)

print("Length of my chunks:", len(text_chunks))

Length of my chunks: 5777


### Download Embedding Model

In [ ]:
def download_hugging_face_embeddings():

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    return embeddings

### Load Embedding Model

In [ ]:
embeddings = download_hugging_face_embeddings()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_9568\2958536756.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

### Verify Embedding Dimension

In [ ]:
query_result = embeddings.embed_query("Hello world")

print("Length:", len(query_result))

Length: 384


In [ ]:
query_result

[-0.03447720408439636,
 0.031023239716887474,
 0.00673496862873435,
 0.026108969002962112,
 -0.03936196118593216,
 -0.16030246019363403,
 0.06692393124103546,
 -0.0064414795488119125,
 -0.047450557351112366,
 0.014758911915123463,
 0.0708753690123558,
 0.05552756413817406,
 0.01919337548315525,
 -0.026251327246427536,
 -0.010109500028192997,
 -0.026940541341900826,
 0.022307470440864563,
 -0.02222665585577488,
 -0.14969269931316376,
 -0.01749308407306671,
 0.007676247972995043,
 0.054352279752492905,
 0.003254473675042391,
 0.03172597661614418,
 -0.0846213549375534,
 -0.0294059906154871,
 0.051595624536275864,
 0.048124030232429504,
 -0.003314792178571224,
 -0.05827920511364937,
 0.04196930304169655,
 0.022210685536265373,
 0.1281888484954834,
 -0.02233896590769291,
 -0.011656301096081734,
 0.06292833387851715,
 -0.032876282930374146,
 -0.09122605621814728,
 -0.031175389885902405,
 0.05269956216216087,
 0.0470348559319973,
 -0.08420302718877792,
 -0.03005620837211609,
 -0.0207447819411

### Store Embeddings in Pinecone

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name="medical-chatbot"
)

### Load Existing Pinecone Index

In [ ]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name="medical-chatbot",
    embedding=embeddings
)

### Perform Similarity Search

In [ ]:
query = "What are Allergies?"

docs = docsearch.similarity_search(query, k=3)

print(docs)

[Document(id='c8dba543-759e-4f79-8319-a045a1a73102', metadata={'author': '', 'creationDate': "D:20041218170002-05'00'", 'creationdate': '2004-12-18T17:00:02-05:00', 'creator': '', 'file_path': 'pdfs\\Medical_Book_Gale Encyclopedia.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': "D:20041218161531-06'00'", 'moddate': '2004-12-18T16:15:31-06:00', 'page': 129.0, 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'pdfs\\Medical_Book_Gale Encyclopedia.pdf', 'subject': '', 'title': '', 'total_pages': 637.0, 'trapped': ''}, page_content='reaction. Allergic rhinitis is characterized by an itchy,\nrunny nose, often with a scratchy or irritated throat due\nto post-nasal drip. Inflammation of the thin membrane\ncovering the eye (allergic conjunctivitis) causes redness,\nirritation, and increased tearing in the eyes. Asthma caus-\nes wheezing, coughing, and shortness of breath. Symp-\ntoms of food allergies depend on the tissues most sensi-\ntive to the allergen and whether the allergen spread

### Create Custom Prompt Template

In [ ]:
prompt_template = """
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know.
Don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

### Create Prompt Object

In [ ]:
PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

### Chain Type Arguments

In [ ]:
chain_type_kwargs = {"prompt": PROMPT}

### Load Llama 2 GGUF Model

In [ ]:
llm = CTransformers(
    model="model/llama-2-7b-chat.Q4_K_M.gguf",
    model_type="llama",
    config={
        'max_new_tokens': 512,
        'temperature': 0.5
    }
)

### Create Retrieval QA Chain

In [ ]:
from langchain_classic.chains import RetrievalQA

In [ ]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(search_kwargs={'k': 2}),
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs
)

### Run Medical Chatbot

In [ ]:
while True:

    user_input = input("Input Prompt: ")

    if user_input == "exit":
        print("Exiting...")
        break

    result = qa({"query": user_input})

    print("Response:", result["result"])

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_9568\2321566791.py:9: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa({"query": user_input})


Response: Diabetes is caused by a combination of not making enough insulin (insulin deficiency) and/or the insulin that is made not working properly (insulin resistance).
Exiting...


In [7]:
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://www.msdmanuals.com/professional/hematology-and-oncology/anemias-caused-by-deficient-erythropoiesis/anemia-of-chronic-disease"
]

loader = WebBaseLoader(urls)
docs = loader.load()

print(docs[0].page_content[:2000])

USER_AGENT environment variable not set, consider setting it to identify your requests.


Anemia of Chronic Disease - Hematology - MSD Manual Professional Editionhoneypot linkskip to main contentProfessionalConsumerMSD ManualProfessional VersionMEDICAL TOPICSRESOURCESDRUG INFOCOMMENTARYPROCEDURESQUIZZESABOUT USMEDICAL TOPICSRESOURCES <Anemias Caused by Deficient ErythropoiesisAnemia of Chronic Disease(Anemia of Chronic Inflammation)ByGloria F. Gerber, MD, Johns Hopkins School of Medicine, Division of HematologyReviewed ByAshkan Emadi, MD, PhD, West Virginia University School of Medicine, Robert C. Byrd Health Sciences
CenterReviewed/Revised Modified Mar 2025v969267View Patient EducationAnemia of chronic disease is a multifactorial anemia. Diagnosis generally requires the presence of a chronic inflammatory condition, such as infection, autoimmune disease, kidney disease, or cancer. It is characterized by a microcytic or normocytic anemia and low reticulocyte count. Values for serum iron and transferrin are typically low, while the serum ferritin value can be normal or elevat

In [8]:
import requests
from bs4 import BeautifulSoup

url = "https://www.msdmanuals.com/professional/hematology-and-oncology/anemias-caused-by-deficient-erythropoiesis/anemia-of-chronic-disease"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

text = soup.get_text(separator=" ", strip=True)

print(text[:5000])

Anemia of Chronic Disease - Hematology - MSD Manual Professional Edition honeypot link skip to main content Professional Consumer MSD Manual Professional Version MEDICAL TOPICS RESOURCES DRUG INFO COMMENTARY PROCEDURES QUIZZES ABOUT US MEDICAL TOPICS RESOURCES < Anemias Caused by Deficient Erythropoiesis Anemia of Chronic Disease (Anemia of Chronic Inflammation) By Gloria F. Gerber , MD , Johns Hopkins School of Medicine, Division of Hematology Reviewed By Ashkan Emadi , MD, PhD , West Virginia University School of Medicine, Robert C. Byrd Health Sciences
Center Reviewed/Revised Modified Mar 2025 v969267 View Patient Education Anemia of chronic disease is a multifactorial anemia. Diagnosis generally requires the presence of a chronic inflammatory condition, such as infection, autoimmune disease, kidney disease, or cancer. It is characterized by a microcytic or normocytic anemia and low reticulocyte count. Values for serum iron and transferrin are typically low, while the serum ferritin